<a href="https://colab.research.google.com/github/qkrsj/LG_aimers/blob/main/W8A8%20GPTQ/LGAimers_Phase2_Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import

In [ ]:
!pip install llmcompressor

In [ ]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# Now these imports will see the version installed by llmcompressor
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier
from llmcompressor.modifiers.awq import AWQModifier

In [ ]:
from huggingface_hub import snapshot_download

# 허깅페이스에서 코랩 로컬 디스크(/content)로 직접 다운로드
model_path = snapshot_download(
    repo_id="LGAI-EXAONE/EXAONE-4.0-1.2B",
    local_dir="./base_model",
    ignore_patterns=["*.msgpack", "*.h5"] # 불필요한 파일 제외
)

# Setting

In [ ]:
MODEL_ID = "./base_model"
OUT_DIR  = "./model"

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 256
MAX_SEQUENCE_LENGTH = 512

# Quantization
SCHEME = "W4A16"
TARGETS = ["Linear"]
IGNORE  = ["embed_tokens", "lm_head"]

# Model Loads

In [ ]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
)

print("[INFO] 모델/토크나이저 로드 완료")

# Dataset Loads & Preprocess

In [ ]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False)
    }

ds = ds.map(preprocess)

print("[INFO] 데이터 전처리 완료")

# Quantization

## GPTQ

In [ ]:
# Quantization
SCHEME = "W8A8"
TARGETS = ["Linear"]
IGNORE  = ["embed_tokens", "lm_head"]

print(f"[INFO] GPTQ 시작 (scheme={SCHEME}, samples={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH})...")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] GPTQ 완료")

L4 기준, GPTQ 양자화 기법 적용 시 6분 소요

## AWQ

In [ ]:
print(f"[INFO] AWQ 시작 (scheme={SCHEME}, samples={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH})...")

recipe = [
    AWQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] AWQ 완료")

# Model Save

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"[INFO] 모델 저장 완료: {OUT_DIR}")

In [ ]:
import os
from pathlib import Path

def get_dir_size(path, extension=".safetensors"):
    """특정 확장자를 가진 파일들의 총 용량을 GB 단위로 반환"""
    total_size = 0
    for file in Path(path).rglob(f"*{extension}"):
        total_size += os.path.getsize(file)
    return total_size / (1024**3)  # GB 단위 변환

# 1. 경로 설정 (기존 모델 경로를 적어주세요)
ORIGINAL_MODEL_PATH = "./base_model" # 혹은 "LGAI-EXAONE/EXAONE-4.0-1.2B"

# 2. 용량 계산
original_size = get_dir_size(ORIGINAL_MODEL_PATH)
quantized_size = get_dir_size(OUT_DIR)

# 3. 결과 출력
print("-" * 30)
print(f"📊 용량 비교 결과 (Unit: GB)")
print(f"· 원본 모델: {original_size:.2f} GB")
print(f"· 양자화 모델: {quantized_size:.2f} GB")
print(f"· 압축률: {(1-quantized_size/original_size)*100:.1f}% 절감")
print("-" * 30)

if quantized_size > original_size:
    print("⚠️ 경고: 양자화 후 용량이 더 커졌습니다. save_compressed=True 설정을 다시 확인하세요.")

# Submission

In [ ]:
zip_name = "baseline_submit"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

print(f"[INFO] 생성 완료: {zip_name}.zip")

# 추론 엔진 적용 - vLLM

In [ ]:
!pip install vllm
from vllm import LLM, SamplingParams

prompts = [
    [{"role":"user", "content":"Explain how wonderful you are"}],
    [{"role":"user", "content":"너가 얼마나 대단한지 설명해 봐"}]
]
sampling_params = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=256)

llm = LLM(model="EXAONE-4.0-1.2B-GPTQ")

outputs = llm.chat(prompts, sampling_params)

for output in outputs:
    print("###########")
    print(output.outputs[0].text)
    print()

# Evaluation

In [ ]:
!pip install lm-eval[vllm]
!pip install vllm
!pip install hf_transfer

In [ ]:
# 1. 경로 설정 (방금 저장한 양자화 모델 경로)
MODEL_PATH = OUT_DIR # 본인의 OUT_DIR 경로로 수정하세요.

!lm_eval --model vllm \
    --model_args pretrained={MODEL_PATH},gpu_memory_utilization=0.8,enable_thinking=False,max_gen_toks=2048 \
    --tasks gsm8k \
    --limit 512 \
    --output_path ./results \
    --apply_chat_template \
    --batch_size auto

In [ ]:
# 1. 경로 설정 (방금 저장한 양자화 모델 경로)
MODEL_PATH = MODEL_ID # 본인의 OUT_DIR 경로로 수정하세요.

!lm_eval --model vllm \
    --model_args pretrained={MODEL_PATH},gpu_memory_utilization=0.8,enable_thinking=False,max_gen_toks=2048 \
    --tasks gsm8k \
    --limit 512 \
    --output_path ./results \
    --apply_chat_template \
    --batch_size auto